In [1]:
import os, sys, json, re, zipfile
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score, classification_report
)

KAGGLE_BASE = '/kaggle/input/datasets/gauravdongre1710/legal-ner-data1'
WORK_DIR    = '/kaggle/working'

print("✅ Libraries loaded")
print(f"   PyTorch : {torch.__version__}")
print(f"   GPU     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Name    : {torch.cuda.get_device_name(0)}")

# label map
LABEL2ID = {"CONTINUATION": 0, "BOUNDARY": 1}
ID2LABEL = {0: "CONTINUATION", 1: "BOUNDARY"}
print(f"\n   Labels  : {ID2LABEL}")

✅ Libraries loaded
   PyTorch : 2.9.0+cu126
   GPU     : True
   Name    : Tesla T4

   Labels  : {0: 'CONTINUATION', 1: 'BOUNDARY'}


In [2]:
from datasets import load_dataset
import random
random.seed(42)
np.random.seed(42)

print("Loading LEDGAR...")
ledgar      = load_dataset("lex_glue", "ledgar", verification_mode='no_checks')
ledgar_data = ledgar['train']
print(f"✅ LEDGAR loaded: {len(ledgar_data)} clauses")

# ── build sentence-level boundary dataset ──
# Strategy:
#   Each LEDGAR example = one complete clause
#   First sentence of clause  → BOUNDARY     (label=1)
#   Other sentences           → CONTINUATION (label=0)

print("\nBuilding boundary dataset from LEDGAR...")

sentences  = []
labels     = []
skipped    = 0

# spaCy for sentence splitting
import spacy
nlp = spacy.load("en_core_web_sm")
nlp.disable_pipes(["tagger", "ner", "lemmatizer"])

MAX_CLAUSES = 15000  # process 15k clauses

for i, example in enumerate(ledgar_data):
    if i >= MAX_CLAUSES:
        break

    text = example['text'].strip()
    if len(text) < 30:
        skipped += 1
        continue

    # split into sentences
    doc   = nlp(text[:1000])  # limit per clause
    sents = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 10]

    if len(sents) < 2:
        skipped += 1
        continue

    # first sentence = BOUNDARY
    sentences.append(sents[0])
    labels.append(1)

    # remaining = CONTINUATION
    for sent in sents[1:]:
        sentences.append(sent)
        labels.append(0)

    if (i+1) % 2000 == 0:
        print(f"  Processed {i+1} clauses → "
              f"{len(sentences)} sentences "
              f"(B:{sum(labels)} C:{len(labels)-sum(labels)})")

print(f"\n✅ Dataset built")
print(f"   Total sentences : {len(sentences)}")
print(f"   BOUNDARY        : {sum(labels)}  ({100*sum(labels)/len(labels):.1f}%)")
print(f"   CONTINUATION    : {len(labels)-sum(labels)}  ({100*(len(labels)-sum(labels))/len(labels):.1f}%)")
print(f"   Skipped         : {skipped}")

Loading LEDGAR...


README.md: 0.00B [00:00, ?B/s]

ledgar/train-00000-of-00001.parquet:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

ledgar/test-00000-of-00001.parquet:   0%|          | 0.00/3.31M [00:00<?, ?B/s]

ledgar/validation-00000-of-00001.parquet:   0%|          | 0.00/3.44M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

✅ LEDGAR loaded: 60000 clauses

Building boundary dataset from LEDGAR...
  Processed 2000 clauses → 2834 sentences (B:1055 C:1779)
  Processed 4000 clauses → 5560 sentences (B:2069 C:3491)
  Processed 10000 clauses → 13797 sentences (B:5132 C:8665)
  Processed 12000 clauses → 16507 sentences (B:6142 C:10365)
  Processed 14000 clauses → 19228 sentences (B:7163 C:12065)

✅ Dataset built
   Total sentences : 20636
   BOUNDARY        : 7682  (37.2%)
   CONTINUATION    : 12954  (62.8%)
   Skipped         : 7318


In [4]:

CUAD_PATTERN = re.compile(
    r'\n\s*((?:\d+\.)+\d*|\d+|(?:Section|Article|SECTION|ARTICLE)\s*\d+)'
    r'[.\s]+([A-Z][A-Za-z -]{2,60})',
    re.MULTILINE
)

# synthetic contract boundaries from known patterns
BOUNDARY_EXAMPLES = [
    "1. Definitions",
    "2. Payment Terms",
    "3. Confidentiality",
    "4. Intellectual Property Rights",
    "5. Termination",
    "6. Governing Law",
    "7. Indemnification",
    "8. Limitation of Liability",
    "9. Dispute Resolution",
    "10. Force Majeure",
    "Section 1. Scope of Services",
    "Section 2. Term and Renewal",
    "Article I. General Provisions",
    "Article II. Representations and Warranties",
    "WHEREAS, the parties desire to enter into this Agreement",
    "NOW THEREFORE, in consideration of the mutual covenants",
    "IN WITNESS WHEREOF, the parties have executed this Agreement",
    "1.1 Confidential Information means any non-public information",
    "2.1 The Company shall pay Licensee within thirty days",
    "3.1 Either party may terminate upon written notice",
]

CONTINUATION_EXAMPLES = [
    "The parties agree to keep all information confidential.",
    "This obligation shall survive the termination of the Agreement.",
    "All payments shall be made in United States Dollars.",
    "The Company shall not be liable for indirect damages.",
    "Such notice shall be given in writing to the other party.",
    "This Agreement shall be binding upon the successors of both parties.",
    "The provisions of this section shall survive termination.",
    "Each party represents that it has full authority to enter this Agreement.",
    "Time is of the essence with respect to all dates in this Agreement.",
    "Neither party may assign this Agreement without prior written consent.",
]

# add augmented boundary/continuation examples (5x each for balance)
for ex in BOUNDARY_EXAMPLES * 5:
    sentences.append(ex)
    labels.append(1)

for ex in CONTINUATION_EXAMPLES * 5:
    sentences.append(ex)
    labels.append(0)

print(f"After augmentation:")
print(f"   Total     : {len(sentences)}")
print(f"   BOUNDARY  : {sum(labels)}")
print(f"   CONT.     : {len(labels)-sum(labels)}")

# ── shuffle and split 80/10/10 ──
indices = list(range(len(sentences)))
random.shuffle(indices)

n_train = int(len(indices) * 0.80)
n_val   = int(len(indices) * 0.10)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train+n_val]
test_idx  = indices[n_train+n_val:]

train_data = {"text": [sentences[i] for i in train_idx],
              "label": [labels[i] for i in train_idx]}
val_data   = {"text": [sentences[i] for i in val_idx],
              "label": [labels[i] for i in val_idx]}
test_data  = {"text": [sentences[i] for i in test_idx],
              "label": [labels[i] for i in test_idx]}

print(f"\nSplit complete")
print(f"   Train : {len(train_data['text'])}")
print(f"   Val   : {len(val_data['text'])}")
print(f"   Test  : {len(test_data['text'])}  ← FROZEN")

After augmentation:
   Total     : 20936
   BOUNDARY  : 7882
   CONT.     : 13054

Split complete
   Train : 16748
   Val   : 2093
   Test  : 2095  ← FROZEN


In [5]:
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded")

# build HuggingFace datasets
train_hf = Dataset.from_dict(train_data)
val_hf   = Dataset.from_dict(val_data)
test_hf  = Dataset.from_dict(test_data)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation     = True,
        max_length     = 128,      # sentences are short — 128 is enough
        padding        = False,    # DataCollator handles padding
    )

print("Tokenizing...")
train_tok = train_hf.map(tokenize, batched=True, remove_columns=["text"])
val_tok   = val_hf.map(tokenize,   batched=True, remove_columns=["text"])
test_tok  = test_hf.map(tokenize,  batched=True, remove_columns=["text"])

print(f"✅ Tokenization complete")
print(f"   Train : {len(train_tok)}")
print(f"   Val   : {len(val_tok)}")
print(f"   Test  : {len(test_tok)}")

# verify sample
s = train_tok[0]
print(f"\n   Sample keys     : {list(s.keys())}")
print(f"   input_ids len   : {len(s['input_ids'])}")
print(f"   label           : {s['label']} ({ID2LABEL[s['label']]})")

Loading tokenizer: nlpaueb/legal-bert-base-uncased


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Tokenizer loaded
Tokenizing...


Map:   0%|          | 0/16748 [00:00<?, ? examples/s]

Map:   0%|          | 0/2093 [00:00<?, ? examples/s]

Map:   0%|          | 0/2095 [00:00<?, ? examples/s]

✅ Tokenization complete
   Train : 16748
   Val   : 2093
   Test  : 2095

   Sample keys     : ['label', 'input_ids', 'token_type_ids', 'attention_mask']
   input_ids len   : 31
   label           : 1 (BOUNDARY)


# Training the model

In [6]:
print("Loading Legal-BERT for sequence classification...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = 2,
    id2label   = ID2LABEL,
    label2id   = LABEL2ID,
)
model.config.hidden_dropout_prob          = 0.2
model.config.attention_probs_dropout_prob = 0.2
print(f"✅ Model loaded — {model.num_parameters():,} parameters")

# ── weighted loss for class imbalance ──
# BOUNDARY is minority class — upweight it
boundary_weight = (len(labels) - sum(labels)) / sum(labels)
print(f"\n   Class imbalance ratio : {boundary_weight:.2f}x")
print(f"   BOUNDARY weight       : {boundary_weight:.2f}")
print(f"   CONTINUATION weight   : 1.00")

device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = torch.tensor([1.0, boundary_weight], dtype=torch.float).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs,
                     return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy" : accuracy_score(labels, preds),
        "f1"       : f1_score(labels, preds, average="macro"),
        "precision": precision_score(labels, preds, average="macro"),
        "recall"   : recall_score(labels, preds, average="macro"),
        "f1_boundary": f1_score(labels, preds, pos_label=1, average="binary"),
    }

total_steps  = (len(train_tok) // 32) * 5
warmup_steps = int(total_steps * 0.1)

args = TrainingArguments(
    output_dir                  = f'{WORK_DIR}/segmenter_checkpoints',
    num_train_epochs            = 5,
    learning_rate               = 2e-5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    warmup_steps                = warmup_steps,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_boundary",
    greater_is_better           = True,
    save_total_limit            = 1,
    fp16                        = True,
    report_to                   = "none",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model            = model,
    args             = args,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print(f"\n=== TRAINING PLAN ===")
print(f"   Train examples  : {len(train_tok)}")
print(f"   Val examples    : {len(val_tok)}")
print(f"   Batch size      : 32 per GPU × 2 GPUs = 64 per step")
print(f"   Total steps     : {total_steps}")
print(f"   Warmup steps    : {warmup_steps}")
print(f"\nStarting training...")

trainer.train()
print("\n Training complete")

Loading Legal-BERT for sequence classification...


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


 Training complete


# evaluating it

In [7]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

# ── 1. Validation Set ──
print("=" * 60)
print("VALIDATION SET EVALUATION")
print("=" * 60)
val_preds    = trainer.predict(val_tok)
val_pred_ids = np.argmax(val_preds.predictions, axis=-1)
val_labels   = val_preds.label_ids

print(classification_report(
    val_labels, val_pred_ids,
    target_names=["CONTINUATION", "BOUNDARY"]
))

# ── 2. Test Set ──
print("=" * 60)
print("TEST SET EVALUATION  ← FINAL SCORE")
print("=" * 60)
test_preds    = trainer.predict(test_tok)
test_pred_ids = np.argmax(test_preds.predictions, axis=-1)
test_labels   = test_preds.label_ids

print(classification_report(
    test_labels, test_pred_ids,
    target_names=["CONTINUATION", "BOUNDARY"]
))

test_f1_boundary = f1_score(test_labels, test_pred_ids,
                             pos_label=1, average="binary")
test_acc         = accuracy_score(test_labels, test_pred_ids)

# ── 3. Confusion Matrix ──
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(test_labels, test_pred_ids)
tn, fp, fn, tp = cm.ravel()
print(f"\n                   Predicted")
print(f"                   CONT.    BOUNDARY")
print(f"Actual CONT.     {tn:6d}    {fp:6d}")
print(f"Actual BOUNDARY  {fn:6d}    {tp:6d}")
print(f"\n  Correctly found boundaries : {tp}")
print(f"  Missed boundaries          : {fn}")
print(f"  False alarms               : {fp}")

# ── 4. Sample Predictions ──
print("\n" + "=" * 60)
print("SAMPLE PREDICTIONS ON REAL CONTRACT SENTENCES")
print("=" * 60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

samples = [
    ("1. Scope of Services: The Service Provider agrees...", 1),
    ("The services include software development and support.", 0),
    ("2. Payment Terms: The Client agrees to pay INR 12,00,000", 1),
    ("Late payments shall accrue interest at 1.5% per month.", 0),
    ("3. Confidentiality: Both parties agree to maintain", 1),
    ("All information shall be kept strictly confidential.", 0),
    ("9. Governing Law: This Agreement shall be governed by India", 1),
    ("Any disputes shall be subject to courts of Mumbai.", 0),
    ("WHEREAS the parties desire to enter into this Agreement", 1),
    ("This obligation shall survive termination of the Agreement.", 0),
]

print(f"\n{'Sentence':52s} TRUE      PRED      CONF  ")
print("-" * 85)

correct = 0
for text, true in samples:
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs   = torch.softmax(outputs.logits[0], dim=-1)
    pred_id = torch.argmax(probs).item()
    conf    = probs[pred_id].item()

    true_str = ID2LABEL[true]
    pred_str = ID2LABEL[pred_id]
    match    = "✅" if pred_id == true else "❌"
    if pred_id == true:
        correct += 1

    print(f"{text[:52]:52s} {true_str:9s} {pred_str:9s} "
          f"{conf:.3f} {match}")

print(f"\n  Sample accuracy: {correct}/{len(samples)}")

# ── 5. Save Decision ──
print("\n" + "=" * 60)
print("SAVE DECISION")
print("=" * 60)
print(f"  Test Accuracy    : {test_acc:.4f}")
print(f"  Test Boundary F1 : {test_f1_boundary:.4f}")

if test_f1_boundary >= 0.83:
    print(f"\n  ✅ SAVE — Boundary F1 {test_f1_boundary:.4f} ≥ 0.83")
elif test_f1_boundary >= 0.75:
    print(f"\n  ⚠️  BORDERLINE — F1 {test_f1_boundary:.4f}")
else:
    print(f"\n  ❌ DO NOT SAVE — F1 {test_f1_boundary:.4f} < 0.75")

VALIDATION SET EVALUATION


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


              precision    recall  f1-score   support

CONTINUATION       0.94      0.83      0.88      1292
    BOUNDARY       0.77      0.91      0.83       801

    accuracy                           0.86      2093
   macro avg       0.85      0.87      0.86      2093
weighted avg       0.87      0.86      0.86      2093

TEST SET EVALUATION  ← FINAL SCORE


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


              precision    recall  f1-score   support

CONTINUATION       0.93      0.82      0.87      1347
    BOUNDARY       0.73      0.89      0.81       748

    accuracy                           0.85      2095
   macro avg       0.83      0.86      0.84      2095
weighted avg       0.86      0.85      0.85      2095

CONFUSION MATRIX

                   Predicted
                   CONT.    BOUNDARY
Actual CONT.       1104       243
Actual BOUNDARY      80       668

  Correctly found boundaries : 668
  Missed boundaries          : 80
  False alarms               : 243

SAMPLE PREDICTIONS ON REAL CONTRACT SENTENCES

Sentence                                             TRUE      PRED      CONF  
-------------------------------------------------------------------------------------
1. Scope of Services: The Service Provider agrees... BOUNDARY  BOUNDARY  0.988 ✅
The services include software development and suppor CONTINUATION CONTINUATION 0.840 ✅
2. Payment Terms: The Client agree

# saving model

In [ ]:
import os, json

SAVE_PATH = f'{WORK_DIR}/segmenter'
os.makedirs(SAVE_PATH, exist_ok=True)

# ── save model + tokenizer ──
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(" Model and tokenizer saved")

# ── save segmenter_config.json ──
seg_config = {
    "model_name"    : "Legal Clause Segmenter",
    "base_model"    : "nlpaueb/legal-bert-base-uncased",
    "version"       : "2.0.0",
    "type"          : "bert_boundary_classifier",
    "labels"        : ID2LABEL,
    "label2id"      : LABEL2ID,
    "num_labels"    : 2,
    "max_length"    : 128,
    "threshold"     : 0.5,
    "input"         : "single sentence string",
    "output"        : "BOUNDARY or CONTINUATION",
    "trained_on"    : "LEDGAR 15k clauses + augmentation",
    "performance"   : {
        "val_boundary_f1"  : 0.83,
        "test_boundary_f1" : 0.8053,
        "test_accuracy"    : 0.8458,
        "sample_accuracy"  : "10/10 on real Indian contract"
    }
}

with open(f'{SAVE_PATH}/segmenter_config.json', 'w') as f:
    json.dump(seg_config, f, indent=2)
print("✅ segmenter_config.json saved")

# ── verify all files ──
print(f"\n=== FILES IN segmenter/ ===")
for fname in os.listdir(SAVE_PATH):
    size = os.path.getsize(f'{SAVE_PATH}/{fname}') / 1024 / 1024
    print(f"  {fname:35s}  {size:.2f} MB")

# ── upload to HuggingFace ──
print("\nUploading to HuggingFace...")
from huggingface_hub import HfApi, login

login(token="")  #  paste your HF token

api = HfApi()
api.create_repo(
    repo_id   = "Devil1710/Legal-Clause-Segmenter",
    repo_type = "model",
    exist_ok  = True
)

api.upload_folder(
    folder_path = SAVE_PATH,
    repo_id     = "Devil1710/Legal-Clause-Segmenter",
    repo_type   = "model"
)

print(" Uploaded to HuggingFace")
print("   URL: https://huggingface.co/Devil1710/Legal-Clause-Segmenter")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Model and tokenizer saved
✅ segmenter_config.json saved

=== FILES IN segmenter/ ===
  segmenter_config.json                0.00 MB
  tokenizer.json                       0.67 MB
  tokenizer_config.json                0.00 MB
  config.json                          0.00 MB
  model.safetensors                    417.67 MB

Uploading to HuggingFace...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Uploaded to HuggingFace
   URL: https://huggingface.co/Devil1710/Legal-Clause-Segmenter


In [9]:
import json, os

SAVE_PATH = f'{WORK_DIR}/segmenter'
os.makedirs(SAVE_PATH, exist_ok=True)

metrics = {
    "model"        : "Devil1710/Legal-Clause-Segmenter",
    "base_model"   : "nlpaueb/legal-bert-base-uncased",
    "trained_on"   : "LEDGAR 15k clauses + augmentation",
    "dataset_sizes": {
        "train"    : 16748,
        "val"      : 2093,
        "test"     : 2095
    },
    "training_epochs": [
        {"epoch":1, "train_loss":"No log", "val_loss":0.4388, "accuracy":0.7840, "f1":0.7815, "f1_boundary":0.7578},
        {"epoch":2, "train_loss":0.5248,   "val_loss":0.3563, "accuracy":0.8423, "f1":0.8374, "f1_boundary":0.8090},
        {"epoch":3, "train_loss":0.5248,   "val_loss":0.3501, "accuracy":0.8442, "f1":0.8414, "f1_boundary":0.8201},
        {"epoch":4, "train_loss":0.2910,   "val_loss":0.3630, "accuracy":0.8533, "f1":0.8501, "f1_boundary":0.8282},
        {"epoch":5, "train_loss":0.2910,   "val_loss":0.3766, "accuracy":0.8614, "f1":0.8577, "f1_boundary":0.8347},
    ],
    "best_epoch"   : 5,
    "validation": {
        "accuracy"        : 0.8614,
        "macro_f1"        : 0.8577,
        "boundary_f1"     : 0.8347,
        "boundary_precision": 0.77,
        "boundary_recall" : 0.91,
        "continuation_f1" : 0.88,
    },
    "test": {
        "accuracy"          : 0.8458,
        "macro_f1"          : 0.8400,
        "boundary_f1"       : 0.8053,
        "boundary_precision": 0.73,
        "boundary_recall"   : 0.89,
        "continuation_f1"   : 0.87,
        "true_positives"    : 668,
        "false_positives"   : 243,
        "false_negatives"   : 80,
        "true_negatives"    : 1104,
    },
    "sample_test"  : {
        "sentences_tested" : 10,
        "correct"          : 10,
        "accuracy"         : "10/10 on real Indian contract"
    },
    "class_weights": {
        "CONTINUATION" : 1.00,
        "BOUNDARY"     : 1.66
    },
    "hyperparameters": {
        "learning_rate"  : 2e-5,
        "epochs"         : 5,
        "batch_size"     : 32,
        "max_length"     : 128,
        "weight_decay"   : 0.01,
        "warmup_ratio"   : 0.10,
        "dropout"        : 0.2,
        "fp16"           : True,
        "early_stopping" : "patience=2"
    }
}

with open(f'{SAVE_PATH}/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("✅ training_metrics.json saved")
print(f"\n=== KEY METRICS ===")
print(f"  Val  Boundary F1 : {metrics['validation']['boundary_f1']}")
print(f"  Test Boundary F1 : {metrics['test']['boundary_f1']}")
print(f"  Test Accuracy    : {metrics['test']['accuracy']}")
print(f"  Sample test      : {metrics['sample_test']['accuracy']}")

# also upload to HuggingFace alongside model
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj = f'{SAVE_PATH}/training_metrics.json',
    path_in_repo    = "training_metrics.json",
    repo_id         = "Devil1710/Legal-Clause-Segmenter",
    repo_type       = "model"
)
print(f"\n✅ Uploaded to HuggingFace")
print(f"   https://huggingface.co/Devil1710/Legal-Clause-Segmenter")


✅ training_metrics.json saved

=== KEY METRICS ===
  Val  Boundary F1 : 0.8347
  Test Boundary F1 : 0.8053
  Test Accuracy    : 0.8458
  Sample test      : 10/10 on real Indian contract

✅ Uploaded to HuggingFace
   https://huggingface.co/Devil1710/Legal-Clause-Segmenter


# More Test